# Understanding the relationships among characteristics of notable people

"Notability" is a social construct that lacks a precise, universally-accepted definition. The BHHT data contain four main attributes describing the notable people -- sex, occupation, birth era, and region. These characteristics can be used to understand the social construction of notability in different contexts. For example, we can consider how the joint distribution of sex and occupation varies geographically (by region) and over time.

The characteristics of notable people in the BHHT data are [nominal](https://en.wikipedia.org/wiki/Nominal_category) (except birth year, which can be recoded if desired to an ordinal "birth era" variable). We can thus represent the data as a 4-way [contingency table](https://en.wikipedia.org/wiki/Contingency_table) (sex x occupation x birth era x region). This contingency table reflects the joint distribution of the four characteristics in the population of interest.


In [42]:
import pandas as pd
import numpy as np
from pathlib import Path
import prince
import matplotlib.pyplot as plt

Modify the path below to point to the data file.


In [43]:
pa = Path("./")

The entire dataset has records for around 2.3 million people. MCA is fast so there is no need to subset the data.


In [51]:
ca = ["birth", "death", "gender", "un_region", "level1_main_occ", "name"]
df = pd.read_csv(
    pa / Path("cross-verified-database.csv.gz"), usecols=ca, encoding="latin-1"
)
df.shape

(2291817, 6)

Rename the variables so that they fit better as labels on plots.


In [52]:
df = df.rename({"level1_main_occ": "occ", "gender": "sex", "un_region": "reg"}, axis=1)
df = df[["birth", "death", "occ", "sex", "reg", "name"]].dropna()

We will focus on people who lived after 1500.


In [54]:
# df = df.query("birth <= 500")
# df.query("birth <= 1000").query("birth > 0").groupby("reg").count()
df.head()
# print(df.shape)
# df.head()

# df.query("birth <= 1500").query("birth > 100").groupby("reg").count()

,birth,death,occ,sex,reg,name
0,1932.0,1990.0,Culture,Male,Europe,Claus_Hammel
1,1860.0,1927.0,Culture,Male,Europe,Karel_MatÄj_Äapek-Chod
4,1912.0,1977.0,Leadership,Female,Europe,Wiltraut_Rupp-von_BrÃ¼nneck
5,1928.0,2016.0,Culture,Male,America,Gillett_Griffin
6,1818.0,1894.0,Discovery/Science,Male,Europe,Joseph_Dienger


Remove very infrequent or difficult to interpret categories


In [27]:
df = df.loc[df.occ != "Other", :]
df = df.loc[df.occ != "Missing", :]
df = df.loc[df.sex != "Other", :]

Create a "century of birth" variable, make it a string so that it is interpreted as a nominal variable.


In [33]:
df["era"] = df["birth"].round(-2).astype(int)
df["birth"] = df["birth"].astype(int)
# df = df.drop("birth", axis=1)
# df.loc[:, "era"] = ["%d" % x for x in df.era]
df.head()

,birth,occ,sex,reg,name,era
0,1932,Culture,Male,Europe,Claus_Hammel,1900
1,1860,Culture,Male,Europe,Karel_MatÄj_Äapek-Chod,1900
2,1971,Culture,Male,Europe,Florian_Eichinger,2000
3,1983,Culture,Male,Europe,Florian_Jahr,2000
4,1912,Leadership,Female,Europe,Wiltraut_Rupp-von_BrÃ¼nneck,1900


We won't need the names anymore so we create a copy of the data that omits them.


We will aim to understand notability by studying the contingency table below.


In [34]:
tab = dx.groupby(["sex", "reg", "occ", "era"]).size().unstack().fillna(0)
tab

era                               1500.0   1600.0   1700.0   1800.0    1900.0  \
sex    reg     occ                                                              
Female Africa  Culture               0.0      1.0      0.0      9.0     640.0   
               Discovery/Science     0.0      1.0      0.0      4.0     205.0   
               Leadership            1.0      4.0     12.0     32.0     664.0   
               Sports/Games          0.0      0.0      0.0      0.0      94.0   
       America Culture               2.0      3.0     44.0   1100.0   21909.0   
               Discovery/Science     0.0      7.0     16.0    411.0    5684.0   
               Leadership            8.0     24.0     79.0    833.0    7246.0   
               Sports/Games          0.0      0.0      1.0     11.0    2124.0   
       Asia    Culture               7.0     25.0     24.0     73.0    3426.0   
               Discovery/Science     1.0      1.0      0.0     16.0     762.0   
               Leadership           35.0    102.0     81.0    145.0    1784.0   
               Sports/Games          0.0      1.0      0.0      0.0     463.0   
       Europe  Culture              87.0    411.0    954.0   5203.0   35578.0   
               Discovery/Science    13.0     54.0     93.0    549.0    8253.0   
               Leadership          415.0    952.0   1070.0   1916.0   10881.0   
               Sports/Games          0.0      4.0      2.0     42.0    5043.0   
       Oceania Culture               0.0      0.0      0.0     79.0    1625.0   
               Discovery/Science     0.0      0.0      0.0     34.0     690.0   
               Leadership            0.0      0.0      0.0    107.0     869.0   
               Sports/Games          0.0      0.0      0.0      2.0     583.0   
Male   Africa  Culture               3.0      6.0      9.0     65.0    2562.0   
               Discovery/Science     6.0     15.0     19.0     98.0    1312.0   
               Leadership           13.0     49.0     81.0    522.0    6798.0   
               Sports/Games          0.0      0.0      0.0     15.0    2665.0   
       America Culture               7.0     77.0    232.0   4558.0   63792.0   
               Discovery/Science    21.0    126.0    228.0   3654.0   31530.0   
               Leadership           73.0    461.0   2254.0  31952.0   71402.0   
               Sports/Games          0.0      1.0      6.0    386.0   47832.0   
       Asia    Culture              91.0    272.0    261.0    719.0   14686.0   
               Discovery/Science    45.0     99.0    127.0    376.0    7447.0   
               Leadership          516.0    830.0    708.0   2193.0   21709.0   
               Sports/Games          1.0     13.0     23.0     72.0    6165.0   
       Europe  Culture            2107.0   7698.0  10952.0  35066.0  138895.0   
               Discovery/Science  1765.0   4324.0   6978.0  26824.0   85234.0   
               Leadership         4024.0  13258.0  18615.0  58104.0  128416.0   
               Sports/Games         25.0     35.0    158.0   2856.0   82886.0   
       Oceania Culture               0.0      0.0      1.0    400.0    3753.0   
               Discovery/Science     0.0      0.0      0.0    510.0    2628.0   
               Leadership            0.0      0.0      7.0   2562.0    8657.0   
               Sports/Games          1.0      0.0      0.0    328.0   15205.0   

era                                 2000.0  
sex    reg     occ                          
Female Africa  Culture              2160.0  
               Discovery/Science     294.0  
               Leadership           1441.0  
               Sports/Games         3640.0  
       America Culture             32551.0  
               Discovery/Science    3292.0  
               Leadership           7665.0  
               Sports/Games        15302.0  
       Asia    Culture             14558.0  
               Discovery/Science     767.0  
               Leadership           2573.0  
            

## Pearson residuals

One way to gain some insight from a multi-way contingency table is to form Pearson residuals. These residuals have the form (observed - expected) / SD(observed), where "observed" and "expected" are the observed and expected cell counts for each cell in the contingency table. The "expected" cell count is obtained under an independence model, and the standard deviation SD(observed) is calculated using the Poisson approximation to the distribution of a Bernoulli trial with small success probability. The Pearson residuals help identify where the observed counts are furthest (in statistical terms) from what would be expected if the attributes were distributed randomly over the observations.


In [ ]:
long = tab.stack().reset_index().rename(columns={0: "count"})
n = long["count"].sum()
vx = ["sex", "reg", "occ", "era"]
vp = ["%s_p" % x for x in vx]
for v in vx:
    long[v + "_p"] = long.groupby(v)["count"].transform(np.sum) / n
long["exp"] = n * long[vp].prod(1)
long["chi2_resid"] = (long["count"] - long["exp"]) / np.sqrt(long["exp"])
long = long.sort_values(by="chi2_resid")
long.sort_values("chi2_resid")

Below is a plot of the order statistics of the chi square residuals.


In [ ]:
plt.grid(True)
plt.xlabel("Rank")
plt.ylabel("Chi squared residual")
plt.plot(np.sort(long["chi2_resid"]))

## Multiple Correspondence Analysis

Multiple Correspondence Analysis (MCA) is a type of factor analysis for categorical data. A common use of MCA is to produce biplots that can be used to visualize the joint distribution of several categorical variables. Here we use MCA to understand the relationships among the contingency table of characteristics of the subjects in the BHHT data.

Below we fit factors to the data using multiple correspondence analysis (MCA).


In [ ]:
mca = prince.MCA(n_components=4)
mca = mca.fit(dx)

We can make an interactive plot of the column coordinates (there are too many objects to plot):


In [ ]:
mca.plot(dx, show_row_markers=False, show_row_labels=False)

We can make a more informative static version of this plot by coloring the levels of the same parent variable in a common color, and by connecting the points that correspond to ordered variables.


In [ ]:
cols = {"occ": "orange", "sex": "purple", "reg": "lime", "era": "navy"}


def mca_plot(mca, df, cols, jx, jy):
    cc = mca.column_coordinates(df)
    xmin, xmax = cc.iloc[:, jx].min(), cc.iloc[:, jx].max()
    d = xmax - xmin
    xmin -= 0.1 * d
    xmax += 0.1 * d
    ymin, ymax = cc.iloc[:, jy].min(), cc.iloc[:, jy].max()
    d = ymax - ymin
    ymin -= 0.1 * d
    ymax += 0.1 * d

    plt.clf()
    plt.grid(True)
    for k in cols.keys():
        cx = cc[cc.index.str.startswith(k)]
        if k == "era":
            plt.plot(cx.iloc[:, jx], cx.iloc[:, jy], "-", color=cols[k])
        for i in range(cx.shape[0]):
            plt.text(
                cx.iloc[i, jx],
                cx.iloc[i, jy],
                cx.index[i],
                color=cols[k],
                ha="center",
                va="center",
            )
    plt.xlabel("Component %d" % (jx + 1))
    plt.ylabel("Component %d" % (jy + 1))
    plt.xlim(xmin, xmax)
    plt.ylim(ymin, ymax)

The most informative projection of the columns is spanned by the first two factors, as plotted below:


In [ ]:
mca_plot(mca, dx, cols, 0, 1)

Recall that the angle between two vectors corresponding to categories of different variables encodes the correlation between the indicators for those variable categories in the Burt matrix. This is illustrated by a few examples below.


In [ ]:
np.corrcoef(dx.occ == "Culture", dx.sex == "Female")

In [ ]:
np.corrcoef(dx.occ == "Leadership", dx.sex == "Female")

In [ ]:
np.corrcoef(dx.reg == "Oceania", dx.occ == "Sports/Games")

In [ ]:
np.corrcoef(dx.occ == "Leadership", dx.era == "1800")

Below we plot factors 2 and 3 (these are the third and fourth factors since Python counts from zero). These capture an independent projection of the data from factors 0 and 1.


In [ ]:
mca_plot(mca, dx, cols, 2, 3)

In an MCA plot, information about the variables is encoded both in the angles between variable scores, and in the magnitudes of the variable scores. Variables that are uncorrelated with all other variables have very short magnitudes. To demonstrate this, we create a variable that is independent of the others and include it in the MCA.


In [ ]:
dx1 = dx.copy()
dx1["fake"] = np.random.choice([0, 1], dx.shape[0])
cols1 = cols.copy()
cols1["fake"] = "red"

mca1 = prince.MCA(n_components=4)
mca1 = mca1.fit(dx1)
mca1.transform(dx1)
mca_plot(mca1, dx1, cols1, 0, 1)